# EDA + Limpieza básica del dataset

Objetivo:
- Entender el dataset
- Analizar la calidad del texto
- Detectar problemas (nulos, ruido, longitud, encoding residual)
- Aplicar limpieza ligera SIN perder información

⚠️ Nota:
No se eliminan duplicados en esta etapa.

In [44]:
import pandas as pd
import re

In [45]:
df = pd.read_csv("../data/processed/data_fixed_v2md.csv")
df.head()

,user_id,tweet_id,tweet_text,class
0,user0001,0d3ed29586ce,Cheesecake saludable sin azúcar y sin lactosa ...,control
1,user0002,c3cf897a495b,ser como ellas ♡♡\n #HastaLosHuesos,anorexia
2,user0003,5041d85c45c6,"Comida Real o , la clave para estar más sana, ...",control
3,user0004,d18285d3c7ec,Entre el cambio de hora y la bajada de las #te...,control
4,user0005,4d81892f3217,Hace mucho tiempo no sentía mi cuerpo tan frío,anorexia


In [46]:
# Información general

print("Shape:", df.shape)
print("\nColumnas:")
print(df.columns.tolist())

print("\nTipos de datos:")
print(df.dtypes)

Shape: (1500, 4)

Columnas:
['user_id', 'tweet_id', 'tweet_text', 'class']

Tipos de datos:
user_id       str
tweet_id      str
tweet_text    str
class         str
dtype: object


In [47]:
df.isnull().sum()

user_id       0
tweet_id      0
tweet_text    0
class         0
dtype: int64

In [48]:
text_cols = df.select_dtypes(include=["object"]).columns.tolist()
text_cols

/var/folders/zx/phs8kxv920b3hn1m7b_f4m_w0000gn/T/ipykernel_48747/4175219668.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include=["object"]).columns.tolist()


['user_id', 'tweet_id', 'tweet_text', 'class']

In [49]:
df["tweet_text_len"] = df["tweet_text"].astype(str).str.len()
df["tweet_text_len"].describe()

count    1500.000000
mean      142.991333
std        78.337831
min        17.000000
25%        89.000000
50%       124.000000
75%       214.250000
max       438.000000
Name: tweet_text_len, dtype: float64

In [50]:
df[df["tweet_text_len"] < 10]["tweet_text"].head(20)

Series([], Name: tweet_text, dtype: str)

In [51]:
df["tweet_text"].sample(20, random_state=42)

1116    "Es el nuevo gurú de las top models, ¿sabes po...
1368         Lo que necesitas para comenzar.  #Motivación
422     Festivo: día en el que puedes entrenar, irte a...
413           ¿Que huesos te gustan más? #proana #thinspo
451     Inspiración princesas, no abandonéis :)  #ana ...
861     Otro día más para entrenar duro y conseguir el...
1063    No es solo deporte, es un estilo de vida. \n#S...
741              A tomar mi te verde  con canela y laurel
1272    Se puede vomitar en el colegio >.< pero es\n\n...
259     Buenos días y #FelizMartes . Hoy toca luchar, ...
888     It's Your Lucky Day .\nNosotros estaremos, par...
1432    Hoy creo que estoy con más hambre que nunca, n...
712     Si puedo!! Lo se no decaigo!! Solo 6 kilos mas...
1382    Verga!\nAyer salí con mis amigas y trague como...
70                Necesito vomitar todo lo que me comi :(
1268    La mujer soñada. asi toda  con todo en su luga...
1491    Con tu participación podemos seguir apoyando a...
607         Pu

In [52]:
bad_pattern = re.compile(r"[ÃÂâ]")

def has_suspicious_text(x):
    return isinstance(x, str) and bool(bad_pattern.search(x))

df["tweet_text"].apply(has_suspicious_text).sum()

np.int64(0)

In [53]:
df[df["tweet_text"].apply(has_suspicious_text)]["tweet_text"].head(10)

Series([], Name: tweet_text, dtype: str)

In [54]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return text
    
    # convertir a minúsculas
    text = text.lower()
    
    # quitar URLs
    text = re.sub(r"http\S+|www\S+", "", text)
    
    # quitar menciones (@usuario)
    text = re.sub(r"@\w+", "", text)

    #conservar las palabras del hashtag
    text = re.sub(r"#(\w+)", r"\1", text)
    
    # normalizar espacios
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()

In [55]:
df["tweet_text_clean"] = df["tweet_text"].apply(clean_text)

In [56]:
comparison = pd.DataFrame({
    "before": df["tweet_text"].head(10),
    "after": df["tweet_text_clean"].head(10)
})

comparison

,before,after
0,Cheesecake saludable sin azúcar y sin lactosa ...,cheesecake saludable sin azúcar y sin lactosa ...
1,ser como ellas ♡♡\n #HastaLosHuesos,ser como ellas ♡♡ hastaloshuesos
2,"Comida Real o , la clave para estar más sana, ...","comida real o , la clave para estar más sana, ..."
3,Entre el cambio de hora y la bajada de las #te...,entre el cambio de hora y la bajada de las tem...
4,Hace mucho tiempo no sentía mi cuerpo tan frío,hace mucho tiempo no sentía mi cuerpo tan frío
5,Ser gorda esta mal,ser gorda esta mal
6,"Si en un restaurante, la pretensión es el ingr...","si en un restaurante, la pretensión es el ingr..."
7,"Puaa increible, nose como permiten esas pagina...","puaa increible, nose como permiten esas pagina..."
8,"Perder una batalla, no significa que la guerra...","perder una batalla, no significa que la guerra..."
9,Cuando la ley no responde nuestras elecciones...,cuando la ley no responde nuestras elecciones ...


In [57]:
df["tweet_text_clean"].sample(20, random_state=42)

1116    "es el nuevo gurú de las top models, ¿sabes po...
1368           lo que necesitas para comenzar. motivación
422     festivo: día en el que puedes entrenar, irte a...
413             ¿que huesos te gustan más? proana thinspo
451     inspiración princesas, no abandonéis :) ana mi...
861     otro día más para entrenar duro y conseguir el...
1063    no es solo deporte, es un estilo de vida. salt...
741               a tomar mi te verde con canela y laurel
1272    se puede vomitar en el colegio >.< pero es asq...
259     buenos días y felizmartes . hoy toca luchar, p...
888     it's your lucky day . nosotros estaremos, para...
1432    hoy creo que estoy con más hambre que nunca, n...
712     si puedo!! lo se no decaigo!! solo 6 kilos mas...
1382    verga! ayer salí con mis amigas y trague como ...
70                necesito vomitar todo lo que me comi :(
1268    la mujer soñada. asi toda con todo en su lugar...
1491    con tu participación podemos seguir apoyando a...
607           

In [58]:
from collections import Counter

hashtags = []

for text in df["tweet_text"].dropna():
    hashtags.extend(re.findall(r"#(\w+)", text.lower()))

Counter(hashtags).most_common(20)

[('anorexia', 183),
 ('bulimia', 139),
 ('proana', 111),
 ('promia', 76),
 ('ana', 75),
 ('thinspo', 67),
 ('mia', 66),
 ('salud', 56),
 ('gym', 54),
 ('tca', 50),
 ('fit', 45),
 ('saludable', 43),
 ('ed', 40),
 ('comidasana', 39),
 ('nutricion', 37),
 ('healthy', 37),
 ('healthyfood', 32),
 ('thinspiration', 31),
 ('adelgazar', 30),
 ('comida', 28)]

In [59]:
# Exportar datos limpios
df.to_csv("../data/processed/data_cleaned_v2md.csv", index=False)